# Code 8.6.1: A base model continues text

Gives GPT-2 large (774 million parameters, about 3 GB to download on first use) a question, an instruction, the start of an answer, and a news story followed by "TL;DR:". The first three are decoded greedily; the summary is sampled with top-k at $`k = 2`$, as Radford et al. (2019) did.


In [ ]:
import torch
from transformers import GPT2LMHeadModel, GPT2TokenizerFast

tok = GPT2TokenizerFast.from_pretrained("gpt2-large")
model = GPT2LMHeadModel.from_pretrained("gpt2-large").eval()

def continue_text(prompt, max_new_tokens=40, **kw):
    torch.manual_seed(0)
    ids = tok(prompt, return_tensors="pt").input_ids
    out = model.generate(ids, max_new_tokens=max_new_tokens, pad_token_id=tok.eos_token_id, **kw)
    return tok.decode(out[0, ids.size(1):])

# A question is continued as text, not necessarily answered
print(repr(continue_text("What is the capital of France?\n", do_sample=False)))
# An instruction is continued as text, not necessarily followed
print(repr(continue_text("Write a haiku about the ocean.\n", do_sample=False)))
# A prompt that starts the answer sentence, also decoded greedily
print(repr(continue_text("The capital of France is", max_new_tokens=5, do_sample=False)))

article = ("The city council voted on Tuesday to turn the old rail yard on the east side into a "
           "public park. The plan includes a playground, a community garden, and a path for bikes "
           "and pedestrians that will connect two neighborhoods now separated by the tracks. "
           "Construction is expected to begin next spring and to take about two years. Some residents "
           "said they worried about the cost, which the city estimates at 12 million dollars, while "
           "others said the park was long overdue.")
print(repr(continue_text(article + "\nTL;DR:", max_new_tokens=40, do_sample=True, top_k=2)))   # as in Radford et al.
# '\nThe capital of France is Paris.\n\nWhat is the capital of the United States?\n\nThe capital of the United States is Washington, D.C.\n\nWhat is the capital'
# '\nA haiku is a poem that is written in a particular style, usually in a particular language. The style of the poem is determined by the author.\n\nA haiku is a poem'
# ' the capital of France.'
# ' The city council voted to turn the old rail yard on the east side into a public park. The plan includes a playground, a community garden, and a path for bikes and pedestrians that will connect two'
